# SG1: One row per player, describing their behavior

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

players   = pd.read_csv("data/synthetic/synthetic_players.csv", parse_dates=["install_time"])
sessions  = pd.read_csv("data/synthetic/synthetic_sessions.csv", parse_dates=["session_start"])
purchases = pd.read_csv("data/synthetic/synthetic_purchases.csv", parse_dates=["purchase_time"])
END = pd.Timestamp("2026-09-30")
WINDOW = 60

players["days_observable"] = (END - players["install_time"].dt.normalize()).dt.days
pl = players.loc[players["days_observable"] >= WINDOW, ["player_id", "install_time"]]
print("Players observable for 60+ days:", len(pl))

Players observable for 60+ days: 45103


In [2]:
# sessions in each player's first 60 days
s = sessions.merge(pl, on="player_id")
s["day_n"] = (s["session_start"].dt.normalize() - s["install_time"].dt.normalize()).dt.days
s = s[s["day_n"] < WINDOW]

# purchases in each player's first 60 days
pu = purchases.merge(pl, on="player_id")
pu["day_n"] = (pu["purchase_time"].dt.normalize() - pu["install_time"].dt.normalize()).dt.days
pu = pu[pu["day_n"] < WINDOW]

feat = s.groupby("player_id").agg(
    active_days=("day_n", "nunique"),          # how many different days they played
    sessions=("day_n", "size"),                # how many sessions in total
    avg_session_min=("duration_minutes", "mean"),
    last_active_day=("day_n", "max"),          # how long they stayed (0 = only the install day)
)
feat["spend_usd"] = pu.groupby("player_id")["amount_usd"].sum()
feat["spend_usd"] = feat["spend_usd"].fillna(0)   # non-payers spent 0
feat.describe().round(1)

,active_days,sessions,avg_session_min,last_active_day,spend_usd
count,45103.0,45103.0,45103.0,45103.0,45103.0
mean,6.0,10.2,39.9,7.9,1.7
std,10.1,17.3,23.8,15.7,21.9
min,1.0,1.0,1.3,0.0,0.0
25%,1.0,2.0,26.2,0.0,0.0
50%,2.0,3.0,36.4,1.0,0.0
75%,5.0,8.0,46.7,6.0,0.0
max,50.0,97.0,650.7,59.0,1259.8


# SG2: Choose the features and the number of groups

In [3]:
feat.corr().round(2)

,active_days,sessions,avg_session_min,last_active_day,spend_usd
active_days,1.00,0.99,0.0,0.99,0.14
sessions,0.99,1.00,0.0,0.98,0.14
avg_session_min,0.00,0.00,1.0,0.00,-0.00
last_active_day,0.99,0.98,0.0,1.00,0.13
spend_usd,0.14,0.14,-0.0,0.13,1.00


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

FEATURES = ["active_days", "sessions", "last_active_day"]
X = StandardScaler().fit_transform(np.log1p(feat[FEATURES]))

for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    sil = silhouette_score(X, km.labels_, sample_size=5000, random_state=42)
    print(f"k={k}  inertia={km.inertia_:,.0f}  silhouette={sil:.3f}")

k=2  inertia=31,946  silhouette=0.702
k=3  inertia=15,444  silhouette=0.622
k=4  inertia=8,263  silhouette=0.625
k=5  inertia=5,592  silhouette=0.614
k=6  inertia=4,340  silhouette=0.665


# SG3: Create the 3 groups and describe them

In [5]:
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
feat["cluster"] = km.labels_

profile = feat.groupby("cluster").agg(
    players=("active_days", "size"),
    active_days=("active_days", "median"),
    sessions=("sessions", "median"),
    last_active_day=("last_active_day", "median"),
    avg_session_min=("avg_session_min", "median"),
    payer_pct=("spend_usd", lambda x: round((x > 0).mean() * 100, 1)),
    spend_per_player=("spend_usd", "mean"),
)
profile["share_pct"] = (profile["players"] / profile["players"].sum() * 100).round(1)
profile = profile.sort_values("active_days")

# Name the groups from least to most engaged
names = dict(zip(profile.index, ["Drop-offs", "Explorers", "Committed"]))
feat["segment"] = feat["cluster"].map(names)
profile.index = profile.index.map(names)
profile.round(2)

,players,active_days,sessions,last_active_day,avg_session_min,payer_pct,spend_per_player,share_pct
cluster,,,,,,,,
Drop-offs,27838,1.0,2.0,0.0,32.80,0.9,0.26,61.7
Explorers,10422,4.0,7.0,5.0,37.67,2.8,1.43,23.1
Committed,6843,26.0,44.0,41.0,39.53,10.4,7.78,15.2


# SG4: Open the answer key

In [6]:
from sklearn.metrics import adjusted_rand_score

truth = pd.read_csv("data/synthetic/answer_key_player_types.csv")
m = feat.reset_index().merge(truth, on="player_id")

expected = {"Drop-offs": "casual", "Explorers": "regular", "Committed": "core"}
m["predicted_type"] = m["segment"].map(expected)

print(f"Agreement: {(m['predicted_type'] == m['true_type']).mean():.1%}")
print(f"Adjusted Rand Index: {adjusted_rand_score(m['true_type'], m['segment']):.3f}\n")

# For each true type: where did those players end up? (rows add up to 100%)
(pd.crosstab(m["true_type"], m["segment"], normalize="index") * 100).round(1)[["Drop-offs", "Explorers", "Committed"]]

Agreement: 77.7%
Adjusted Rand Index: 0.499



segment,Drop-offs,Explorers,Committed
true_type,,,
casual,88.3,11.7,0.0
core,2.3,10.0,87.7
regular,24.9,51.8,23.3


# SG5: Summary of findings
## Summary: player segmentation (synthetic data)

**Note:** synthetic data modeled on Palia; the hidden player types were used only at the
end, to check the result.

### Approach
- Players described by their **first 60 days** (45,103 players observable that long),
  so older and newer players are compared fairly.
- Clustered on **engagement only** (active days, sessions, last active day): log-transformed
  and standardized, KMeans with k = 3.
- Spending was left out of the clustering and analyzed per segment afterwards; session length
  was left out because it is unrelated to engagement (correlation 0.00).
- k = 3 chosen as a balance between statistics (silhouette 0.62, close to the best)
  and usefulness (a meaningful middle group).

### Segments
| Segment | Share | Typical behavior (first 60 days) | Payers | Spend per player |
|---|---|---|---|---|
| Drop-offs | 62% | 1 day, 2 sessions | 0.9% | $0.26 |
| Explorers | 23% | 4 days over about a week | 2.8% | $1.43 |
| Committed | 15% | 26 days, still active around day 41 | 10.4% | $7.78 |

### Validation against the hidden types
- 78% agreement, Adjusted Rand Index 0.50.
- Casual and core players were recovered with about 88% accuracy; "regular" players split
  across all three segments because their actual behavior varied, not because of the method.

### Implications
- Committed players are worth about 30x a Drop-off in the first 60 days.
- Explorers are the main opportunity: they showed interest for a week before leaving, so better
  mid-game goals or social features could move some of them into the Committed group.

### Limitations
- Synthetic data; the three engagement features are almost perfectly correlated, so the
  segmentation mainly splits players along one engagement axis.

In [7]:
import os
OUT = "data/powerbi"
os.makedirs(OUT, exist_ok=True)

# 1. Player table: one row per player (the central "dimension" table)
dim = players[["player_id", "install_time", "platform", "region"]].copy()
dim["install_date"]  = dim["install_time"].dt.date
dim["install_month"] = dim["install_time"].dt.to_period("M").astype(str)

# Event window vs normal installs (same definition as in the retention notebook)
BIG_RELEASES = ["2025-05-13", "2026-05-12"]
in_window = pd.Series(False, index=dim.index)
for d in BIG_RELEASES:
    start = pd.Timestamp(d)
    in_window |= (dim["install_time"] >= start) & (dim["install_time"] < start + pd.Timedelta(days=21))
dim["install_group"] = np.where(in_window, "Event window", "Normal")

# Segment from the clustering; players observable < 60 days weren't segmented
dim["segment"] = dim["player_id"].map(feat["segment"]).fillna("Not segmented (<60 days)")
dim.drop(columns="install_time").to_csv(f"{OUT}/dim_players.csv", index=False)

# 2. Segment profile
profile.rename_axis("segment").reset_index().round(2).to_csv(f"{OUT}/segment_profile.csv", index=False)

print("Saved:", sorted(os.listdir(OUT)))
print(dim["segment"].value_counts())
print(dim["install_group"].value_counts())

Saved: ['churn_timing.csv', 'daily_kpis.csv', 'dim_players.csv', 'fact_purchases.csv', 'ltv_curve.csv', 'retention_cohorts.csv', 'retention_curve.csv', 'retention_event_vs_normal.csv', 'segment_profile.csv', 'whale_curve.csv']
segment
Drop-offs                   27838
Explorers                   10422
Committed                    6843
Not segmented (<60 days)     4897
Name: count, dtype: int64
install_group
Normal          42899
Event window     7101
Name: count, dtype: int64
